# Space Effects on Looking without Seeing
In this notebook we examine the spatial aspects of our paradigm.
<br><br>
First, we look at the spatial distribution of targets and distractors in our task to verify that they are distributed uniformly across the display and across target categories.<br>
Then, we examine _HITS_ ans _MISSES_ across space to see if there are any systematic patterns in their distribution.<br>
Lastly, we look at the spatial distribution of Looking without Seeing (LWS; i.e., $P[\text{miss} | \text{on target}]$) to see if there are any patterns that predict where LWS occurs.

In [1]:
import warnings
from typing import Literal

import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from pygam import LogisticGAM, te, f

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

import config as cnfg
from analysis.helpers.read_data import read_data
from analysis.helpers.funnels.build_funnels import build_event_classification_funnel

pio.renderers.default = "notebook"      # "notebook" or "browser"
warnings.filterwarnings("ignore", category=UserWarning, message=".*FigureCanvasAgg is non-interactive.*")

In [18]:
JET_COLORSCALE = [
    [0.0, "rgba(255, 255, 255, 0)"],
    [0.001, "rgba(255, 255, 255, 0)"],
    [0.01, "rgba(0, 0, 255, 1)"],
    [0.25, "rgba(0, 255, 255, 1)"],
    [0.5, "rgba(255, 255, 0, 1)"],
    [0.9, "rgba(255, 0, 0, 1)"],
]

def plot_screen_fixed_heatmaps(
        df: pd.DataFrame,
        group_col: str,
        mode: Literal["density", "probability"] = "density",
        title: str = None,
        screen_width: int = 1920,
        screen_height: int = 1080,
        colorscale: str | list = JET_COLORSCALE,
) -> go.Figure:
    """
    Generates heatmaps covering the full screen area (0 to screen_width/height).
    Empty areas will show the low-end of the colorscale (0) rather than gaps.
    """
    categories = sorted(df[group_col].unique().tolist())
    k = len(categories)

    # Grid logic
    cols = 2 if k in [2, 4] else 3
    sub_rows = int(np.ceil(k / cols))
    total_rows = sub_rows + 1

    fig = make_subplots(
        rows=total_rows, cols=cols,
        specs=[[{"colspan": cols}] + [None] * (cols - 1)] + [[{}] * cols for _ in range(sub_rows)],
        subplot_titles=["Overall"] + [str(cat).replace("_", " ") for cat in categories],
        row_heights=[0.5] + [(0.5 / sub_rows)] * sub_rows,
        vertical_spacing=0.05
    )

    # Overall trace
    _add_fixed_grid_trace(
        fig, df, 1, 1,
        nbinsx=screen_width // 20, nbinsy=screen_height // 20,
        width=screen_width, height=screen_height,
        colorscale=colorscale,
        mode=mode,
    )

    # Subgroup traces
    for i, cat in enumerate(categories):
        subset = df[df[group_col] == cat]
        _add_fixed_grid_trace(
            fig, subset,
            row=(i // cols) + 2, col=(i % cols) + 1,
            nbinsx=screen_width // 20, nbinsy=screen_height // 20,
            width=screen_width, height=screen_height,
            colorscale=colorscale,
            mode=mode,
        )

    _apply_layout(fig, screen_width, screen_height, total_rows, title)
    return fig

def _add_fixed_grid_trace(
        fig, data, row, col, nbinsx, nbinsy, width, height, colorscale, mode
):
    """ Calculates a 2D density map anchored to the absolute screen boundaries. """
    # Define the edges for the entire screen
    x_edges = np.linspace(0, width, nbinsx + 1)
    y_edges = np.linspace(0, height, nbinsy + 1)

    # Compute 2D Histogram
    # np.histogram2d automatically handles points outside the range by ignoring them,
    # and points inside the range are counted. Bins with no data are 0.
    z_total, _, _ = np.histogram2d(data["x"], data["y"], bins=[x_edges, y_edges])
    if mode.lower() == "density":
        z_final = z_total
    elif mode.lower() == "probability":
        success_subset = data[data["is_lws"] == 1]
        z_success, _, _ = np.histogram2d(success_subset["x"], success_subset["y"], bins=[x_edges, y_edges])
        with np.errstate(divide='ignore', invalid='ignore'):
            z_final = np.true_divide(z_success, z_total)
    else:
        raise ValueError(f"Unknown mode: {mode}")
    z_final[z_total == 0] = np.nan

    # Add the trace
    trace = go.Heatmap(
        z=z_final.T,   # plotly Heatmap expects [Y, X])
        x=np.linspace(0, width, nbinsx),
        y=np.linspace(0, height, nbinsy),
        colorscale=colorscale,
        zmin=0, zmax=max(np.max(z_final), 1),
        zsmooth="best", connectgaps=False,
        hoverinfo='z+x+y', showscale=False,
    )
    fig.add_trace(trace, row=row, col=col)

def _apply_layout(fig, width, height, n_rows, title):
    fig_width = 1000
    aspect_ratio = height / width
    fig_height = fig_width * aspect_ratio * (n_rows * 0.7)

    # Set hard axes limits to the screen dimensions
    fig.update_xaxes(range=[0, width], showgrid=True, mirror=True, showline=True, linecolor="black")
    fig.update_yaxes(range=[height, 0], showgrid=True, mirror=True, showline=True, linecolor="black")

    fig.update_layout(
        width=fig_width, height=int(fig_height),
        title=dict(text=title, x=0.5, xanchor='center'),
        template="plotly_white",
        margin=dict(l=40, r=40, t=80, b=40)
    )

## Visualize Spatial Counts
We visualize how many instances of each "event" (target existence, missed target, target visits, fixations) occur across the screen. This gives us a visual representation of the spatial distribution of these "events" across the display, and allows us to check for any systematic biases in their distribution (e.g., more targets appearing on the left side of the screen, or more missed targets in the center).

In [3]:
loaded_data = read_data(cnfg.OUTPUT_PATH)
targets = loaded_data.targets
idents = loaded_data.identifications
fixations = loaded_data.fixations
visits = loaded_data.visits
del loaded_data

In [4]:
funnel_results = build_event_classification_funnel(
    cnfg.OUTPUT_PATH, "lws", "visit", exclude="invalid_trials"
)

### Targets

In [19]:
plot_screen_fixed_heatmaps(targets, "category", title="Target Distribution")

### Missed Targets

In [20]:
missed_targets = (
    idents
    .loc[idents["identification_category"] == "miss", ["subject", "trial", "target"]]
    .merge(targets, on=["subject", "trial", "target"])
)
plot_screen_fixed_heatmaps(missed_targets, "category", title="Missed Target Distribution")

### Fixations

In [21]:
plot_screen_fixed_heatmaps(fixations, "subject", title="Fixation Distribution")

### Visits

In [22]:
plot_screen_fixed_heatmaps(visits, "subject", title="Target-Visit Distribution")

### LWS Probability
The ratio between the number of LWS visits and the total number of visits, within each spatial bin

In [23]:
plot_screen_fixed_heatmaps(
    funnel_results, "target_category", title="LWS-Visit Probability", mode="probability"
)

## Statistical Analysis of $P[\text{LWS}]$ across Space
We want to answer the question _"Are the locations/regions on the screen that are more likely to elicit LWS?"_.<br>
More technically, we want to see if $P[\text{miss} | \text{on target}]$ varies across different "bins" of the screen, while controlling for subject - across all trials and within each trial category.
<br><br>
To test the statistical significance, we fit a **Generalized Additive Model (GAM)** - a generalization of GLMMs that can model non-linear relationships between predictors and the outcome variable. In our case, we can use a GAM to model the probability of LWS as a smooth function of the spatial coordinates (x, y) of the visits, while including random effects for subjects to account for individual baseline differences (intercepts). The model is specified as follows:
$$\text{logit}(P[\text{miss} | \text{on target}]) = C(\text{trial\_category}) + te(x, y) + (1 | \text{subject})$$
Where:
- $C(\text{trial\_category})$ is a categorical predictor of trial category ("color", "noise", "bw")
- $te(x, y)$ is a tensor product smooth term that models the non-linear interaction between the spatial coordinates (x, y) of the target-visit's center
- $(1 | \text{subject})$ is a random intercept for each subject to account for individual differences in baseline LWS probability.


### Prepare Data

In [ ]:
# scale the visits' (X, Y) coordinates:
scaler = MinMaxScaler()
funnel_results[["scaled_x", "scaled_y"]] = scaler.fit_transform(funnel_results[["x", "y"]])

# store trial categories as number labels
funnel_results["trial_category_code"] = funnel_results["trial_category"].cat.codes

### Analyze with `pyGAM`
Package `pyGAM` is the Python-native package for fitting Generalized Additive Models.<br>
**NOTE:** this package currently doesn't support _hierarchical_ models (i.e., the $(1 | \text{subject} term)$. Instead, we specify $subject$ as a regular (categorical) predictor.
<br><br>
#### IMPORTANT NOTE
There is a known bug with `pyGAM`: github.com/dswah/pyGAM/issues/163

In [ ]:
# TODO: re-run the statistical analysis once `pygam` fixes the p-value issues (github.com/dswah/pyGAM/issues/163)

predictors = funnel_results[["subject", "trial_category_code", "scaled_x", "scaled_y"]].copy()
response = funnel_results["is_lws"]

gam = LogisticGAM(f(1) + te(2, 3, by=1) + f(0)).fit(predictors, response)
gam.summary()

### Analyze with R-package `mgcv`
The _R_ package `mgcv` is the industry-standard for running GAMs, and it supports our use-case of hierarchical models (allowing subject-specific intercepts). We use Python's R-engine package `rpy2`, which lets us run R code from a Python script.

In [ ]:
import os

os.environ['R_HOME'] = r'C:\Program Files\R\R-4.5.2'
os.environ['PATH'] = r'C:\Program Files\R\R-4.5.2\bin\x64;' + os.environ['PATH']

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, numpy2ri
from rpy2.robjects.conversion import localconverter
from rpy2.robjects.packages import importr

mgcv = importr('mgcv')
base = importr('base')
stats = importr('stats')

In [ ]:
converter = ro.default_converter + pandas2ri.converter + numpy2ri.converter

with localconverter(converter):
    subset = (
        funnel_results[["subject", "trial_category", "scaled_x", "scaled_y", "is_lws"]]
        .astype({"subject": str, "trial_category": str, "scaled_x": float, "scaled_y": float, "is_lws": int})
    )
    rdata = ro.conversion.py2rpy(subset)
    rdata.rx2['subject'] = base.as_factor(rdata.rx2['subject'])
    rdata.rx2['trial_category'] = base.as_factor(rdata.rx2['trial_category'])
    model = mgcv.gam(
        ro.Formula("is_lws ~ trial_category + s(scaled_x, scaled_y) + s(subject, bs='re')"),
        data=rdata,
        family=stats.binomial()
    )

print(base.summary(model))